In [11]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
if CWD.name == "diagnostic":
    DIAGNOSTIC_DIR = CWD
elif (CWD / "diagnostic").is_dir():
    DIAGNOSTIC_DIR = CWD / "diagnostic"
else:
    DIAGNOSTIC_DIR = next((p for p in [CWD, *CWD.parents] if p.name == "diagnostic"), CWD.parent)
if str(DIAGNOSTIC_DIR) not in sys.path:
    sys.path.insert(0, str(DIAGNOSTIC_DIR))


In [ ]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
from scipy.stats import ttest_1samp
from scipy.ndimage import gaussian_filter

import cmocean
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.colors as mcolors

from util.dask_helpers import configure_dask, open_dataset_lazy, persist_if_dask


In [13]:
from configs.initial_land_experiment_config import (
    DEFAULT_DIAG_DIR,
    DEFAULT_FIGURE_DIR,
    DEFAULT_LANDMASK_FILE,
    DEFAULT_SOIL_LAYER_FILE,
    DEFAULT_SOIL_MOISTURE_PLOT,
    get_soil_moisture_data_dict,
)

data_dict = get_soil_moisture_data_dict()
plot_config = DEFAULT_SOIL_MOISTURE_PLOT.copy()
out_path = str(DEFAULT_DIAG_DIR)
fig_path = str(DEFAULT_FIGURE_DIR)

from util.initial_land_support import ensure_initial_land_support_files


In [ ]:
class SoilMoistureDifferencePlotter:
    def __init__(self, data_dict, landmask_file=None, soilayer_file=None,
                 variable='H2OSOI', dz_var='DZSOI', landmask_var='landfrac',
                 depth_cm=10, mask_land=True, confidence_level=0.05,
                 use_mass_units=True, use_dask=True, dask_chunks=None):
        self.data_dict = data_dict
        self.variable = variable
        self.dz_var = dz_var
        self.landmask_var = landmask_var
        self.depth_cm = depth_cm
        self.mask_land = mask_land
        self.significance_threshold = confidence_level
        self.confidence_level = confidence_level
        self.use_mass_units = use_mass_units
        self.water_density = 1000.0
        self.use_dask = use_dask
        self.dask_chunks = dask_chunks or {"time": 1, "levgrnd": -1, "lat": 90, "lon": 180}
        self.dask_client = configure_dask(use_distributed=False) if use_dask else None
        
        self.landmask = None
        if landmask_file and os.path.exists(landmask_file):
            ds_mask = open_dataset_lazy(landmask_file, chunks=self.dask_chunks) if use_dask else xr.open_dataset(landmask_file)
            self.landmask = ds_mask['landmask']
        
            # Detect longitude coordinate name (case-insensitive)
            lon_name = None
            lower_coords = {k.lower(): k for k in self.landmask.coords}
            for test in ['lon', 'longitude']:
                if test in lower_coords:
                    lon_name = lower_coords[test]
                    break
        
            # Convert to 0–360 if needed
            if lon_name:
                lon = self.landmask[lon_name]
                if lon.min() < 0 or lon.max() > 360:
                    new_lon = (lon % 360).sortby(lon)
                    self.landmask = self.landmask.assign_coords({lon_name: new_lon})
                    self.landmask = self.landmask.sortby(lon_name)

    
        self.fixed_dz = None
        if soilayer_file and os.path.exists(soilayer_file):
            ds_dz = open_dataset_lazy(soilayer_file, chunks=self.dask_chunks) if self.use_dask else xr.open_dataset(soilayer_file)
            self.fixed_dz = ds_dz[dz_var]

    def _load_dataset(self, info, year, ensemble=None):
        template = info['template']
        path = info['path']
        filename = template % {'year': year, 'ensemble': ensemble} if ensemble else template % {'year': year}
        full_path = os.path.join(path, filename)
        if not os.path.exists(full_path):
            raise FileNotFoundError(f"[ERROR] File not found: {full_path}")
        ds = open_dataset_lazy(full_path, chunks=self.dask_chunks) if self.use_dask else xr.open_dataset(full_path)

        return ds 

    def normalize_longitude_and_sort(self, ds, lon_name='lon'):
        """
        Normalize longitude in the dataset to [-180, 180] and sort by longitude.
    
        Parameters
        ----------
        ds : xr.DataArray or xr.Dataset
            Input data with a longitude coordinate.
        lon_name : str, default='lon'
            Name of the longitude coordinate.
    
        Returns
        -------
        xr.DataArray or xr.Dataset
            Dataset with normalized and sorted longitude.
        """
        if lon_name in ds.coords:
            lon = ds[lon_name]
            if lon.max() > 180:
                print("[INFO] Normalizing longitude to [-180, 180].")
                ds[lon_name] = ((lon + 180) % 360) - 180
                ds = ds.sortby(lon_name)
        return ds
    
    
    def _wrap_longitude_for_plot(self, da):
        """Append one column to wrap around at 180° for seamless plotting."""
        if 'lon' not in da.coords:
            return da
        if da.lon.max() > 180 or da.lon.min() < -180:
            return da  # already in 0–360 or irregular
        lon = da.lon
        if lon[0] < -179 and lon[-1] > 179:
            wrapped = da.pad(lon=(0, 1), constant_values=np.nan)
            wrapped['lon'][-1] = wrapped['lon'][0] + 360
            return wrapped
        return da
        
    def _integrate_top_layers(self, ds):
        sm = ds[self.variable]

        if self.dz_var in ds:
            dz = ds[self.dz_var]
            if "time" in dz.dims:
                dz = dz.isel(time=0)
        elif self.fixed_dz is not None:
            dz = self.fixed_dz
        else:
            raise ValueError("No valid DZSOI found in dataset or external source.")

        dz_cumsum = dz.cumsum(dim='levgrnd')
        top_mask = dz_cumsum <= (self.depth_cm / 100.0)
        sm_top = sm.where(top_mask)
        dz_masked = dz.where(top_mask)
        if self.use_mass_units:
            sm_mass = sm_top * dz_masked * self.water_density
            integrated = sm_mass.sum(dim='levgrnd')
            integrated.attrs['units'] = 'kg/m2'
        else:
            integrated = (sm_top * dz_masked).sum(dim='levgrnd') / dz_masked.sum(dim='levgrnd')
            integrated.attrs['units'] = 'm3/m3'

        if self.landmask is not None:
            integrated = integrated.where(self.landmask > 0.1)

        return integrated

    def _load_model_ensemble_snapshot(self, model_info, target_date):
        nens = model_info['nens']
        ens_data = []
        for n in range(1, nens + 1):
            ens = f'EN{n:02d}'
            ds = self._load_dataset(model_info, target_date.year, ensemble=ens)
            ds_day = ds.sel(time=target_date)
            sm_top = self._integrate_top_layers(ds_day)
            ens_data.append(sm_top.squeeze())
        return xr.concat(ens_data, dim='ensemble')

    def _load_obs_snapshot(self, obs_info, target_date):
        ds = self._load_dataset(obs_info, target_date.year)
        obs = ds[self.variable].sel(time=target_date)
    
        # Detect longitude coordinate name (case-insensitive)
        lon_name = None
        lower_coords = {k.lower(): k for k in obs.coords}
        for test in ['lon', 'longitude']:
            if test in lower_coords:
                lon_name = lower_coords[test]
                break
                
        # Convert longitude from [-180, 180] to [0, 360] if needed
        if lon_name:
            lon = obs[lon_name]
            if lon.min() < 0 or lon.max() > 360:
                new_lon = (lon % 360).sortby(lon)
                obs = obs.assign_coords({lon_name: new_lon})
                obs = obs.sortby(lon_name)
    
        if self.use_mass_units:
            obs = obs * 1000.0 * self.depth_cm / 100.0
    
        if self.landmask is not None:
            obs = obs.where(self.landmask > 0.1)
    
        return obs

    def _wrap_longitude_for_plot(self, da):
        if 'lon' not in da.coords:
            return da
    
        lon = da.lon
        if (lon[-1] - lon[0]) < 359:  # Avoid double-padding
            da = da.pad(lon=(0, 1), constant_values=np.nan)
            da['lon'][-1] = da['lon'][0] + 360  # Wrap around
        return da
        
    def _compute_significance_mask(self, ensemble_data, obs_mean, alpha=0.05, threshold=None):
        if threshold is None:
            threshold = self.significance_threshold

        diff = ensemble_data - obs_mean
        diff_np = diff.transpose("ensemble", "lat", "lon").compute().values
        tstat, pvals = ttest_1samp(diff_np,
                                   popmean=0.0, axis=0, nan_policy='omit')

        sig_mask = xr.DataArray(pvals < alpha, coords=obs_mean.coords, dims=obs_mean.dims)
        model_mean = ensemble_data.mean(dim='ensemble')
        valid_mask = (abs(model_mean) > threshold) & (abs(obs_mean) > threshold)
        return sig_mask.where(valid_mask)

    def plot_bias_and_stddev_multi_panel(self, model_keys, target_date='2012-01-01',
                                         obs_key='ESA_CCI', bias_levels=None, spread_levels=None,
                                         cmap_bias='RdBu_r', cmap_std='viridis',
                                         figsize=(14.0, 6.6), savepath=None, fontz=9):
        target_date = pd.to_datetime(target_date)
        ncols = len(model_keys)
    
        if bias_levels is None:
            bias_levels = [-0.5, -0.3, -0.1, -0.05, -0.01, 0.01, 0.05, 0.1, 0.3, 0.5]
        if spread_levels is None:
            spread_levels = [0.005, 0.01, 0.02, 0.04, 0.06, 0.08, 0.1]
    
        bias_cmap = mpl.colors.ListedColormap(plt.get_cmap(cmap_bias)(np.linspace(0, 1, len(bias_levels) - 1)))
        bias_norm = mcolors.BoundaryNorm(bias_levels, ncolors=bias_cmap.N)
        spread_cmap = mpl.colors.ListedColormap(plt.get_cmap(cmap_std)(np.linspace(0, 1, len(spread_levels) - 1)))
        spread_norm = mcolors.BoundaryNorm(spread_levels, ncolors=spread_cmap.N)
        bias_label = "Bias (kg m$^{-2}$)" if self.use_mass_units else "Bias (m$^3$ m$^{-3}$)"
        spread_label = "Spread (kg m$^{-2}$)" if self.use_mass_units else "Spread (m$^3$ m$^{-3}$)"
    
        fig, axes = plt.subplots(
            nrows=2, ncols=ncols, figsize=figsize,
            subplot_kw={'projection': ccrs.Robinson()},
            constrained_layout=False
        )
    
        for j, model_key in enumerate(model_keys):
            model_info = self.data_dict[model_key]
            obs_info = self.data_dict[obs_key]
    
            ens_stack = self._load_model_ensemble_snapshot(model_info, target_date)
            obs_field = self._load_obs_snapshot(obs_info, target_date)
    
            if ens_stack.shape[1:] != obs_field.shape:
                ens_stack = ens_stack.interp_like(obs_field)
    
            model_mean = ens_stack.mean(dim='ensemble')
            model_std = ens_stack.std(dim='ensemble')
            bias = model_mean - obs_field
            
            model_mean = self._wrap_longitude_for_plot(model_mean)
            bias = self._wrap_longitude_for_plot(bias)
            model_std = self._wrap_longitude_for_plot(model_std)
            
            model_vals = model_mean.values.flatten()
            obs_vals = obs_field.values.flatten()
            valid = np.isfinite(model_vals) & np.isfinite(obs_vals)
            pcc = np.corrcoef(model_vals[valid], obs_vals[valid])[0, 1]
            rmse_val = float(np.sqrt(((model_vals[valid] - obs_vals[valid]) ** 2).mean()))
    
            # Bias panel
            ax0 = axes[0, j]
            bias_mesh = ax0.contourf(
                bias['lon'], bias['lat'], bias.values,
                levels=bias_levels,
                cmap=bias_cmap,
                norm=bias_norm,
                transform=ccrs.PlateCarree(),
                extend='both'
            )
    
            ax0.set_title(f"{model_key} - {obs_key} (Bias)", fontsize=fontz, pad=8)
            ax0.coastlines(linewidth=0.4)
            ax0.tick_params(labelsize=fontz - 1)
            
            ax0.text(
                0.98, 0.02, f"PCC: {pcc:.2f}\nRMSE: {rmse_val:.2f}",
                transform=ax0.transAxes, fontsize=fontz - 2, va='bottom', ha='right',
                bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.1')
            )
    
            sig_mask = self._compute_significance_mask(ens_stack, obs_field, alpha=self.confidence_level)
            sig_mask.plot.contourf(
                ax=ax0, transform=ccrs.PlateCarree(),
                colors='none', hatches=['..', '..'], 
                add_colorbar=False, 
                add_labels=False,
                linewidths=0
            )
    
            # Spread panel
            ax1 = axes[1, j]
            spread_mesh = model_std.plot.pcolormesh(
                ax=ax1, transform=ccrs.PlateCarree(),
                cmap=spread_cmap,
                norm=spread_norm,
                add_labels=False,
                add_colorbar=False
            )
    
            ax1.set_title(f"{model_key} (Spread)", fontsize=fontz, pad=8)
            ax1.coastlines(linewidth=0.4)
            ax1.tick_params(labelsize=fontz - 1)

            bias_cbar = fig.colorbar(
                bias_mesh, ax=ax0, orientation='horizontal',
                ticks=bias_levels[::2], fraction=0.055, pad=0.08, aspect=24
            )
            bias_cbar.set_label(bias_label, fontsize=fontz - 1, labelpad=2)
            bias_cbar.ax.tick_params(labelsize=fontz - 2, pad=1)

            spread_cbar = fig.colorbar(
                spread_mesh, ax=ax1, orientation='horizontal',
                ticks=spread_levels[::2], fraction=0.055, pad=0.08, aspect=24
            )
            spread_cbar.set_label(spread_label, fontsize=fontz - 1, labelpad=2)
            spread_cbar.ax.tick_params(labelsize=fontz - 2, pad=1)
    
        # Adjust layout after per-panel colorbars are attached.
        plt.subplots_adjust(left=0.035, right=0.985, bottom=0.09, top=0.92, wspace=0.12, hspace=0.46)
    
        plt.show()
    
        if savepath:
            fig.savefig(savepath, dpi=600, bbox_inches='tight')
            print(f"[SAVED] Figure saved to: {savepath}")



In [ ]:
if __name__ == "__main__":
    os.makedirs(fig_path, exist_ok=True)

    landmask_file, soilayer_file = ensure_initial_land_support_files(
        data_dict,
        landmask_file=DEFAULT_LANDMASK_FILE,
        soilayer_file=DEFAULT_SOIL_LAYER_FILE,
        sample_key=plot_config["model_keys"][0],
        target_date=plot_config["target_date"],
    )

    model_keys = plot_config["model_keys"]
    target_date = plot_config["target_date"]
    obs_key = plot_config["obs_key"]
    bias_levels = plot_config["bias_levels"]
    spread_levels = plot_config["spread_levels"]

    combined_savefile = os.path.join(
        fig_path, f"sm_bias_stddev_vs_{obs_key}_{target_date.replace('-', '')}.png"
    )

    plotter = SoilMoistureDifferencePlotter(
        data_dict=data_dict,
        landmask_file=landmask_file,
        soilayer_file=soilayer_file,
        depth_cm=plot_config["depth_cm"],
        mask_land=plot_config["mask_land"],
        confidence_level=plot_config["confidence_level"],
        use_mass_units=plot_config["use_mass_units"],
    )

    plotter.plot_bias_and_stddev_multi_panel(
        model_keys=model_keys,
        target_date=target_date,
        obs_key=obs_key,
        bias_levels=bias_levels,
        spread_levels=spread_levels,
        figsize=plot_config["figsize"],
        fontz=plot_config["fontz"],
        cmap_bias=cmocean.cm.curl,
        cmap_std=cmocean.cm.deep,
        savepath=combined_savefile,
    )
